In [1]:
!pip install numba joblib dask

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 1.2 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.3 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.4 MB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 1.2 MB/s eta 0:00:01
   ----------------------------------- ---- 1.3/1.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.0 MB/s  0:00:01

   ---------- ----------------------------- 1/4 [fsspec]
   ---------- ----------------------------- 1/4 [fsspec]
   ---------- ----------------------------- 1/4 [fsspec]
   ---------- ----------------------------- 1/4 [fsspec]
   ---------- ----------------------------

In [2]:
import numpy as np
import time
from numba import jit
from joblib import Parallel, delayed
import dask.array as da

print("--- 1. Vectorize x^2 + 2x + 1 ---")
x = np.arange(1_000_000)
res_vec = x**2 + 2*x + 1
print("Vectorized calculation complete.\n")

print("--- 2. Numba function to sum squares ---")
@jit(nopython=True)
def sum_squares_numba(arr):
    total = 0.0
    for i in range(len(arr)):
        total += arr[i]**2
    return total

# First call compiles the function (this is why we ignore the first timing)
_ = sum_squares_numba(np.array([1.0, 2.0])) 
print(f"Numba sum of squares: {sum_squares_numba(x.astype(float))}\n")

print("--- 3. Parallel Bootstrap using Joblib ---")
data = np.random.normal(50, 10, 100)

def bootstrap_mean(_):
    sample = np.random.choice(data, size=len(data), replace=True)
    return np.mean(sample)

start = time.time()
boot_means = Parallel(n_jobs=-1)(delayed(bootstrap_mean)(i) for i in range(10000))
print(f"Parallel Bootstrap Time: {time.time() - start:.4f} seconds\n")

print("--- 4. Dask Array for Mean of Large Chunked Array ---")
dask_array = da.random.normal(10, 2, size=(10_000_000,), chunks=(1_000_000,))
dask_mean = dask_array.mean().compute()
print(f"Dask computed mean: {dask_mean:.4f}")

--- 1. Vectorize x^2 + 2x + 1 ---
Vectorized calculation complete.

--- 2. Numba function to sum squares ---
Numba sum of squares: 3.3333283333312755e+17

--- 3. Parallel Bootstrap using Joblib ---
Parallel Bootstrap Time: 3.5059 seconds

--- 4. Dask Array for Mean of Large Chunked Array ---
Dask computed mean: 10.0004


Explain why benchmark timing should exclude Numba's first compilation call.

Answer: Numba uses Just-In-Time (JIT) compilation. The first time a Numba-decorated function is called, it translates the Python code into optimized machine code, which takes significantly longer than the actual execution. Subsequent calls run at native C-like speeds. If you include the first call in your benchmark, it will massively inflate the average execution time, making Numba look slower than it actually is for repeated operations.